# 05 — Predict Departure Drift

**Can we predict how late a bus will be using only the first 15 minutes of ETA data?**

This notebook builds an early-warning regression model on top of the drift snapshots
ingested by notebooks 01–02 and analyzed in notebook 03. The pipeline:

1. Engineers features from the *early observation window* (first 15 min of each trip)
2. Trains a Gradient Boosting Regressor to predict `actual_final_drift`
3. Logs everything to **MLflow** for experiment tracking
4. Visualizes predicted vs actual and feature importance

In [0]:
%pip install mlflow -q
dbutils.library.restartPython()

In [0]:
from pyspark.sql import Window
from pyspark.sql.functions import (
    avg, col, count, first, last, max as spark_max, min as spark_min,
    unix_timestamp, when, hour, dayofweek, lit, row_number, desc,
)
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

In [0]:
dbutils.widgets.text("catalog", "workspace")
dbutils.widgets.text("schema", "cumtd_eta_drift")
dbutils.widgets.text("early_window_minutes", "15")
dbutils.widgets.text("min_snapshots", "10")

catalog = dbutils.widgets.get("catalog").strip()
schema_name = dbutils.widgets.get("schema").strip()
early_window = int(dbutils.widgets.get("early_window_minutes"))
min_snapshots = int(dbutils.widgets.get("min_snapshots"))

database_name = f"{catalog}.{schema_name}" if catalog else schema_name
raw_table = f"{database_name}.raw_departure_snapshots"
metrics_table = f"{database_name}.departure_drift_metrics"

print(f"Source: {raw_table}")
print(f"Early window: {early_window} min  |  Min snapshots: {min_snapshots}")

In [0]:
# ── Feature Engineering ───────────────────────────────────────────────────
# For each trip, extract features from the FIRST N minutes of snapshots
# and predict the final observed drift (last ETA ≈ ground truth).

snaps = (
    spark.table(raw_table)
    .where(col("trip_id").isNotNull())
    .where(col("stop_id").isNotNull())
    .where(col("estimatedDeparture").isNotNull())
    .where(col("scheduledDeparture").isNotNull())
    .withColumn("drift_min",
        (unix_timestamp("estimatedDeparture") - unix_timestamp("scheduledDeparture")) / 60.0)
    .withColumn("obs_hour", hour("ingestion_timestamp"))
    .withColumn("obs_dow", dayofweek("ingestion_timestamp"))
)

trip_w = Window.partitionBy("stop_id", "trip_id", "route_id").orderBy("ingestion_timestamp")
full_w = trip_w.rowsBetween(Window.unboundedPreceding, Window.unboundedFollowing)

snaps = (
    snaps
    .withColumn("snap_num", row_number().over(trip_w))
    .withColumn("first_ts", first("ingestion_timestamp").over(full_w))
    .withColumn("last_drift", last("drift_min", ignorenulls=True).over(full_w))
    .withColumn("total_snaps", count("*").over(full_w))
    .withColumn("mins_since_first",
        (unix_timestamp("ingestion_timestamp") - unix_timestamp("first_ts")) / 60.0)
)

# Early window only
early_features = (
    snaps
    .where(col("total_snaps") >= min_snapshots)
    .where(col("mins_since_first") <= float(early_window))
    .groupBy("stop_id", "trip_id", "route_id", "route_short_name")
    .agg(
        first("obs_hour").alias("hour_of_day"),
        first("obs_dow").alias("day_of_week"),
        first("drift_min").alias("initial_drift"),
        last("drift_min").alias("drift_at_window_end"),
        avg("drift_min").alias("avg_early_drift"),
        spark_max("drift_min").alias("max_early_drift"),
        spark_min("drift_min").alias("min_early_drift"),
        (spark_max("drift_min") - spark_min("drift_min")).alias("early_drift_range"),
        count("*").alias("early_snap_count"),
        first("last_drift").alias("actual_final_drift"),
    )
    .withColumn("drift_velocity",
        (col("drift_at_window_end") - col("initial_drift")) / (col("early_snap_count") + 1))
)

train_pdf = early_features.toPandas()
print(f"Training samples: {len(train_pdf)} trips")
print(f"\nTarget distribution (actual_final_drift):")
print(train_pdf["actual_final_drift"].describe().round(2))

In [0]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.compose import ColumnTransformer

FEATURES = [
    "hour_of_day", "day_of_week",
    "initial_drift", "drift_at_window_end", "avg_early_drift",
    "max_early_drift", "min_early_drift", "early_drift_range",
    "early_snap_count", "drift_velocity",
    "route_short_name",
]
TARGET = "actual_final_drift"

df = train_pdf[FEATURES + [TARGET]].dropna()
X = df[FEATURES]
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f"Train: {len(X_train)}  |  Test: {len(X_test)}")
print(f"Target range: [{y.min():.1f}, {y.max():.1f}] min")

In [0]:
import mlflow
import mlflow.sklearn
from mlflow.models import infer_signature
import numpy as np

# ── Model Training + MLflow ───────────────────────────────────────────
numeric_features = [f for f in FEATURES if f != "route_short_name"]
categorical_features = ["route_short_name"]

preprocessor = ColumnTransformer([
    ("num", StandardScaler(), numeric_features),
    ("cat", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1),
     categorical_features),
])

pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", GradientBoostingRegressor(
        n_estimators=200, max_depth=4,
        learning_rate=0.1, subsample=0.8, random_state=42,
    )),
])

mlflow.set_experiment("/Users/anirudhkonidala@gmail.com/cumtd-eta-drift-prediction")

with mlflow.start_run(run_name="gbr_early_warning_v1") as run:
    pipeline.fit(X_train, y_train)
    y_pred_train = pipeline.predict(X_train)
    y_pred_test = pipeline.predict(X_test)

    train_mae = mean_absolute_error(y_train, y_pred_train)
    test_mae = mean_absolute_error(y_test, y_pred_test)
    test_rmse = np.sqrt(mean_squared_error(y_test, y_pred_test))
    test_r2 = r2_score(y_test, y_pred_test)

    mlflow.log_params({
        "model_type": "GradientBoostingRegressor",
        "n_estimators": 200, "max_depth": 4,
        "learning_rate": 0.1, "early_window_minutes": early_window,
        "n_features": len(FEATURES),
        "train_size": len(X_train), "test_size": len(X_test),
    })
    mlflow.log_metrics({
        "train_mae": train_mae, "test_mae": test_mae,
        "test_rmse": test_rmse, "test_r2": test_r2,
    })

    signature = infer_signature(X_test, y_pred_test)
    mlflow.sklearn.log_model(
        pipeline, "model",
        signature=signature,
        input_example=X_test.iloc[:3],
    )

    print(f"\n{'='*50}")
    print(f"  Early Warning Model Results")
    print(f"{'='*50}")
    print(f"  Train MAE:  {train_mae:.2f} min")
    print(f"  Test MAE:   {test_mae:.2f} min")
    print(f"  Test RMSE:  {test_rmse:.2f} min")
    print(f"  Test R\u00b2:    {test_r2:.3f}")
    print(f"{'='*50}")
    print(f"  MLflow Run: {run.info.run_id}")

In [0]:
# ── Chart: Predicted vs Actual + Feature Importance ───────────────────
ACCENT = "#6366f1"
BG = "#ffffff"

fig, axes = plt.subplots(1, 2, figsize=(16, 6.5), dpi=140)
fig.patch.set_facecolor(BG)

# ── Left: Scatter ──
ax1 = axes[0]
ax1.set_facecolor(BG)

lims = [min(y_test.min(), y_pred_test.min()) - 1, max(y_test.max(), y_pred_test.max()) + 1]
ax1.plot(lims, lims, color="#d1d5db", linewidth=2, linestyle="--", zorder=1)

errors = np.abs(y_test.values - y_pred_test)
scatter = ax1.scatter(
    y_test, y_pred_test,
    c=errors, cmap="RdYlGn_r", s=50, alpha=0.8, edgecolors="white",
    linewidth=0.6, zorder=3, vmin=0, vmax=max(5, np.percentile(errors, 95)),
)
cbar = fig.colorbar(scatter, ax=ax1, shrink=0.8, pad=0.02)
cbar.set_label("Prediction error (min)", fontsize=10, color="#6b7280")
cbar.ax.tick_params(labelsize=9, colors="#6b7280")

ax1.set_xlabel("Actual final drift (min)", fontsize=12, color="#6b7280", labelpad=10)
ax1.set_ylabel("Predicted drift (min)", fontsize=12, color="#6b7280", labelpad=10)
ax1.set_title("Predicted vs Actual Drift", fontsize=16, fontweight="bold", color="#111827", loc="left", pad=12)
ax1.text(0.0, 1.06, f"Test set  \u00b7  MAE = {test_mae:.2f} min  \u00b7  R\u00b2 = {test_r2:.3f}",
         transform=ax1.transAxes, fontsize=11, color="#9ca3af", va="bottom")
ax1.set_xlim(lims)
ax1.set_ylim(lims)
ax1.set_aspect("equal")
ax1.tick_params(labelsize=10, colors="#6b7280", length=0)
for spine in ax1.spines.values():
    spine.set_visible(False)
ax1.grid(alpha=0.25, color="#e5e7eb", linewidth=0.8)

# ── Right: Feature Importance ──
gbr = pipeline.named_steps["model"]
feature_names = numeric_features + categorical_features
importances = gbr.feature_importances_
sorted_idx = np.argsort(importances)
display_names = [n.replace("_", " ").title() for n in np.array(feature_names)[sorted_idx]]

ax2 = axes[1]
ax2.set_facecolor(BG)

alphas_fi = np.linspace(0.35, 1.0, len(sorted_idx))
for i, (name, imp, alpha) in enumerate(zip(display_names, importances[sorted_idx], alphas_fi)):
    ax2.barh(i, imp, color=ACCENT, alpha=alpha, height=0.6, edgecolor="white", linewidth=0.8)
    ax2.text(imp + 0.005, i, f"{imp:.1%}", va="center", fontsize=10, color="#374151", fontweight="bold")

ax2.set_yticks(range(len(display_names)))
ax2.set_yticklabels(display_names, fontsize=10, color="#374151")
ax2.set_title("Feature Importance", fontsize=16, fontweight="bold", color="#111827", loc="left", pad=12)
ax2.text(0.0, 1.06, "What signals predict delay?",
         transform=ax2.transAxes, fontsize=11, color="#9ca3af", va="bottom")
ax2.tick_params(left=False, bottom=False, labelsize=10, colors="#6b7280")
for spine in ax2.spines.values():
    spine.set_visible(False)
ax2.grid(axis="x", alpha=0.25, color="#e5e7eb", linewidth=0.8)

plt.tight_layout()
plt.show()

---
## Results & Takeaways

| Metric | Value |
| --- | --- |
| Test MAE | **2.04 min** |
| Test RMSE | 3.00 min |
| Test R² | 0.067 |
| Training samples | 305 trips (1 evening of data) |

### What the numbers mean

The **MAE of ~2 minutes** means the model's predictions are off by about 2 minutes on average —
useful for a "will this bus be roughly on time or significantly late?" alert. The **low R²** is
expected: we're training on just one evening of data (~305 trips), and most trips have near-zero
drift, making outliers hard to learn.

### Feature importance tells the real story

1. **Route (36%)** — Which bus line you're riding is the single biggest predictor of delay. Some routes are structurally unreliable.
2. **Drift at 15 min (18%)** — If the ETA is already slipping at the 15-minute mark, the bus will likely arrive late.
3. **Hour of day (13%)** — Late-night trips drift differently than rush hour.

This is a **proof of concept on one evening of snapshots**. With a week of data across all stops
(once the Lambda ingests system-wide), the model should improve substantially.

### What’s Next

- Collect multi-day, multi-stop data to grow the training set from 305 → 10K+ trips
- Add weather, event calendar, and headway features
- Deploy as a real-time MLflow Serving endpoint for proactive rider alerts
- Build a classification variant: "Will this bus be >5 min late? Yes/No"